# Week 8 Exercise — RAG Price Estimation Agent

This notebook implements a Retrieval-Augmented Generation (RAG) agent for product price estimation. Product summaries from the dataset are embedded using a sentence-transformer model and stored in a Chroma vector database. When estimating the price of a new product, the agent first retrieves similar products and their known prices from the vector store and uses them as context for the language model. The notebook also includes a baseline agent that estimates prices without retrieval, allowing a comparison between standard LLM predictions and RAG-enhanced predictions.



In [ ]:
!pip install chromadb sentence-transformers litellm datasets gradio python-dotenv huggingface_hub

In [ ]:
import os
import re
import random
import numpy as np
from tqdm import tqdm

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import chromadb

from litellm import completion

import gradio as gr

In [ ]:
dataset = load_dataset("ed-donner/items_lite")

train = list(dataset["train"])
test = list(dataset["test"])

print(f"Train items: {len(train)}")
print(f"Test items: {len(test)}")

In [ ]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
STORE_SIZE = 1000

store_items = random.sample(train, STORE_SIZE)

summaries = [item["summary"] for item in store_items]
prices = [item["price"] for item in store_items]

print("Encoding product summaries...")
embeddings = encoder.encode(summaries, show_progress_bar=True)

chroma_client = chromadb.Client()

collection = chroma_client.create_collection(name="products")

collection.add(
    ids=[str(i) for i in range(STORE_SIZE)],
    embeddings=embeddings.tolist(),
    documents=summaries,
    metadatas=[{"price": p} for p in prices],
)

print("Vector store created.")

In [ ]:
def extract_price(text):
    text = str(text).replace(",", "").replace("$", "")
    match = re.search(r"\d+\.?\d*", text)
    return float(match.group()) if match else 0.0

In [ ]:
def find_similar_products(description, n_results=5):

    query_embedding = encoder.encode([description])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )

    docs = results["documents"][0]
    prices = [m["price"] for m in results["metadatas"][0]]

    return docs, prices

In [ ]:
MODEL = "openrouter/openai/gpt-4o-mini"


def baseline_agent_predict(description):

    prompt = f"""
Estimate the price of this product.

Product description:
{description}

Respond with only a dollar amount.
"""

    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )

    return extract_price(response.choices[0].message.content)

In [ ]:
def rag_agent_predict(description):

    docs, prices = find_similar_products(description)

    context = "Similar products:\n\n"

    for doc, price in zip(docs, prices):
        context += f"{doc[:150]}...\nPrice: ${price:.2f}\n\n"

    prompt = f"""
    Estimate the price of this product.

    Product:
    {description}

    Use the similar products below as reference.

    {context}

    Respond with only a dollar amount.
    """

    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )

    price = extract_price(response.choices[0].message.content)

    return price, prices

In [ ]:
item = test[0]

print(item["summary"])
print("\nActual price:", item["price"])

baseline_price = baseline_agent_predict(item["summary"])
rag_price, _ = rag_agent_predict(item["summary"])

print("\nBaseline prediction:", baseline_price)
print("RAG prediction:", rag_price)

In [ ]:
SIZE = 20

sample = random.sample(test, SIZE)

baseline_errors = []
rag_errors = []

for item in tqdm(sample):

    truth = item["price"]

    baseline = baseline_agent_predict(item["summary"])
    rag, _ = rag_agent_predict(item["summary"])

    baseline_errors.append(abs(baseline - truth))
    rag_errors.append(abs(rag - truth))

print("\nBaseline MAE:", np.mean(baseline_errors))
print("RAG MAE:", np.mean(rag_errors))

In [ ]:
def predict(description):

    baseline = baseline_agent_predict(description)
    rag_price, similar_prices = rag_agent_predict(description)

    docs, _ = find_similar_products(description)

    context = ""

    for doc, price in zip(docs, similar_prices):
        context += f"${price:.2f} — {doc[:80]}...\n"

    return f"${baseline:.2f}", f"${rag_price:.2f}", context


def load_random_item():

    item = random.choice(test)

    return item["summary"], f"${item['price']:.2f}"

In [ ]:
with gr.Blocks(title="RAG Price Estimator") as demo:

    gr.Markdown("# RAG Price Estimation Agent")

    description = gr.Textbox(
        lines=5,
        label="Product Description"
    )

    with gr.Row():
        predict_btn = gr.Button("Estimate Price")
        random_btn = gr.Button("Random Example")

    actual_price = gr.Textbox(label="Actual Price")

    with gr.Row():
        baseline_out = gr.Textbox(label="Baseline Agent")
        rag_out = gr.Textbox(label="RAG Agent")

    context_out = gr.Textbox(label="Retrieved Similar Products", lines=6)

    predict_btn.click(
        fn=predict,
        inputs=description,
        outputs=[baseline_out, rag_out, context_out],
    )

    random_btn.click(
        fn=load_random_item,
        outputs=[description, actual_price],
    )

demo.launch()